# **Deep Learning for Computer Vision**

## **Project 2** - Use of Data Augmentation and Transfer Learning

CRISP-DM Phase 1 to 5

---

Authors: **António Cruz** (140129), **David Isaac** (120064), **Erik Daskalyuk** (120062), **Ivan Magalhães** (106586), **Ricardo Pereira** (120052)

Date: **2025-11-30**

&#160;

# 0. Environment Setup

---

In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
tqdm.pandas()

import cv2
from scipy.stats import entropy, kurtosis, skew
from sklearn.utils import class_weight as sk_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_curve

import tensorflow as tf
from keras import Input, Model, models, regularizers
from keras.optimizers import AdamW
from keras.metrics import AUC, Recall
from keras.applications import EfficientNetV2B0, ResNet50V2
from keras.layers import GlobalAveragePooling2D, Dense, Concatenate, Dropout, Input
from keras import Sequential
from keras.callbacks import ReduceLROnPlateau
from keras.callbacks import EarlyStopping
from keras.models import Model

from keras.callbacks import ModelCheckpoint
from keras.utils import image_dataset_from_directory
from keras.layers import (
    RandomRotation, RandomTranslation, RandomZoom, 
    RandomFlip, RandomBrightness, RandomContrast, GaussianNoise
)

2025-11-28 16:13:27.266486: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-28 16:13:27.311799: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-28 16:13:28.017651: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# Global configuration
SEED = 42                                           # Fixed random seed for reproducibility
DATASET_ROOT = "./chest_xray/"                      # Set the dataset location base path
TRAIN_DIR = DATASET_ROOT + "train"                  # Train samples folder
TEST_DIR = DATASET_ROOT + "test"                    # Test samples folder
CHECKPOINT_PATH = "best_model_checkpoint.keras"     # File path to save the best model
ACTIVE_SCENARIO = "DEVELOPMENT"                     # Active scenario selected: "DEVELOPMENT", "STAGING", "PRODUCTION"

In [3]:
# Set random seeds for reproducibility across libraries
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [4]:
# Helper function to check the active scenario
def scenario(*modes):
    # If in PRODUCTION, everything is allowed
    if ACTIVE_SCENARIO == "PRODUCTION":
        return True
    # Otherwise only allow explicitly listed modes
    return ACTIVE_SCENARIO in modes

# Utility: parse dataset folders into a DataFrame
def load_dataset_df(train_root, test_root):
    """Build a DataFrame with columns: image_path, label, split."""
    records = []

    # Parse training split folders
    for label_name in os.listdir(train_root):
        label_dir = os.path.join(train_root, label_name)
        if not os.path.isdir(label_dir):
            continue

        for f in os.listdir(label_dir):
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                records.append({
                    "image_path": os.path.join(label_dir, f),
                    "label": label_name,
                    "split": "train"
                })

    # Parse test split folders
    for label_name in os.listdir(test_root):
        label_dir = os.path.join(test_root, label_name)
        if not os.path.isdir(label_dir):
            continue

        for f in os.listdir(label_dir):
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                records.append({
                    "image_path": os.path.join(label_dir, f),
                    "label": label_name,
                    "split": "test"
                })

    return pd.DataFrame(records)

In [5]:
# Create DF_ORIGIN, the immutable reference DataFrame
DF_ORIGIN = load_dataset_df(TRAIN_DIR, TEST_DIR)

print("DF_ORIGIN created:")
display(DF_ORIGIN.head())
print(f"Total images: {len(DF_ORIGIN)}")

# DF_VIEW, it will only be used for Business and Data Understanding
DF_VIEW = DF_ORIGIN.copy()

# DF_MODEL, it will only be used for Data Preparation and Modeling
DF_MODEL = DF_ORIGIN.copy()

print("\nDataset copies created:")
print(f"DF_VIEW : {len(DF_VIEW)} rows")
print(f"DF_MODEL: {len(DF_MODEL)} rows")

DF_ORIGIN created:


,image_path,label,split
0,./chest_xray/train/NORMAL/NORMAL2-IM-0911-0001...,NORMAL,train
1,./chest_xray/train/NORMAL/IM-0477-0001.jpeg,NORMAL,train
2,./chest_xray/train/NORMAL/NORMAL2-IM-0974-0001...,NORMAL,train
3,./chest_xray/train/NORMAL/IM-0548-0001.jpeg,NORMAL,train
4,./chest_xray/train/NORMAL/NORMAL2-IM-0917-0001...,NORMAL,train


Total images: 5856

Dataset copies created:
DF_VIEW : 5856 rows
DF_MODEL: 5856 rows


&#160;

# 1. Business Understanding

---

## Scope

This project focuses on developing an AI-powered classification model using transfer learning to analyze chest X-ray images and identify patients with pneumonia. Building on the foundational work of Project 1, this project explores advanced deep learning techniques to improve model performance, generalization, and clinical utility.

The project leverages pre-trained convolutional neural networks (CNNs), such as VGG16, ResNet50, and EfficientNet, to extract meaningful features from chest X-ray images. By fine-tuning these models and applying data augmentation, the goal is to enhance the model's ability to generalize to unseen data while maintaining a strong focus on high recall, ensuring that pneumonia cases are not missed.

The system will:
- Classify chest X-ray images into Normal and Pneumonia categories.
- Enable healthcare professionals to retrospectively identify at-risk patients for targeted preventive care, reducing the risk of pneumonia recurrence and complications.

## Objectives

1. Implement Transfer Learning
   Develop a classification model using pre-trained CNNs (e.g., VGG16, ResNet50, EfficientNet) as a baseline, leveraging their ability to extract high-level features from medical images.

2. Experiment with Data Augmentation
   Apply transformations such as rotation, zoom, and flipping to artificially expand the training dataset, improving model robustness and reducing overfitting.

3. Explore Feature Extraction and Fine-Tuning
   - Feature Extraction: Use pre-trained models as fixed feature extractors, training only the top classifier layers.
   - Fine-Tuning: Unfreeze and retrain later layers of the pre-trained models to adapt them specifically to the chest X-ray classification task.

4. Evaluate Model Performance
   Compare the performance of different architectures and techniques using clinically relevant metrics, including accuracy, precision, recall, F1-score, and AUC-ROC, with an emphasis on recall to ensure that pneumonia cases are not overlooked.

5. Justify the Best-Performing Model
   Provide a detailed analysis of why the selected model (e.g., ResNet50 with fine-tuning) outperforms others, considering trade-offs such as training time, computational resources, and clinical utility.

6. Deliver a Functional and Documented Solution
   Provide a well-documented Python codebase and a comprehensive report detailing the dataset, methodology, model architecture, training process, evaluation results, and the rationale behind the selected model and techniques.

## Clinical and Operational Impact

The project aims to deliver a practical, recall-focused AI model that assists healthcare professionals in identifying patients with a history of pneumonia from historical X-ray records. By enabling targeted preventive interventions, the model supports secondary prevention efforts, reducing the risk of recurrence and improving patient outcomes.

&#160;

# 2. Data Understanding
---

In [6]:
# Global Contrast (std)
def compute_global_std(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.nan
    return float(np.std(img.astype(np.float32)))

# Dynamic Range (max - min)
def compute_dynamic_range(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.nan
    img = img.astype(np.float32)
    return float(np.max(img) - np.min(img))

# Histogram Entropy
def compute_hist_entropy(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.nan
    hist = cv2.calcHist([img], [0], None, [256], [0, 256])
    p = hist.ravel().astype(np.float32)
    p = p / np.sum(p)
    p[p == 0] = 1e-12
    return float(entropy(p, base=2))

# Kurtosis and Skewness
def compute_kurtosis_skew(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.nan, np.nan
    vals = img.astype(np.float32).ravel()
    return float(kurtosis(vals)), float(skew(vals))

# Local Contrast Uniformity (tile std mean & variance)
def compute_local_contrast(path, tiles=8):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.nan, np.nan

    img = img.astype(np.float32)
    h, w = img.shape
    tile_h = h // tiles
    tile_w = w // tiles

    stds = []
    for i in range(tiles):
        for j in range(tiles):
            tile = img[i * tile_h:(i + 1) * tile_h,
                       j * tile_w:(j + 1) * tile_w]
            stds.append(np.std(tile))

    stds = np.array(stds)
    return float(stds.mean()), float(stds.var())

# High-Frequency Content (mean Laplacian)
def compute_high_freq_energy(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.nan
    lap = cv2.Laplacian(img.astype(np.float32), cv2.CV_32F)
    return float(np.mean(np.abs(lap)))

# Geometric Metrics: Centering and Content Analysis
def compute_geometric_metrics(path):
    """
    Compute geometric metrics for augmentation parameter estimation.
    
    Returns:
        centering_offset_x, centering_offset_y, 
        content_width_ratio, content_height_ratio,
        aspect_ratio
    """
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    
    h, w = img.shape
    aspect_ratio = w / h  # ADD THIS
    
    # Find content bounding box (non-black pixels)
    mask = img > 10
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    
    if not (rows.any() and cols.any()):
        return 0.0, 0.0, 1.0, 1.0, aspect_ratio  # UPDATE THIS
    
    y_min, y_max = np.where(rows)[0][[0, -1]]
    x_min, x_max = np.where(cols)[0][[0, -1]]
    
    # Compute content center
    content_center_x = (x_min + x_max) / 2
    content_center_y = (y_min + y_max) / 2
    
    # Compute image center
    image_center_x = w / 2
    image_center_y = h / 2
    
    # Compute centering offsets (normalized by image dimensions)
    offset_x = (content_center_x - image_center_x) / w
    offset_y = (content_center_y - image_center_y) / h
    
    # Compute content size ratios
    content_width = x_max - x_min + 1
    content_height = y_max - y_min + 1
    width_ratio = content_width / w
    height_ratio = content_height / h
    
    return float(offset_x), float(offset_y), float(width_ratio), float(height_ratio), float(aspect_ratio)

## 2.1 Global Contrast

A stable model performance requires a dataset with relatively consistent exposure and contrast characteristics. Large fluctuations in global contrast, whether caused by acquisition variability, compression artifacts, or post-processing, can lead to heterogeneous feature distributions during training. Such variability increases the risk of the model focusing on contrast artifacts rather than clinically relevant patterns. Quantifying global contrast supports the identification of low-quality or overprocessed images that may require correction, filtering, or consistent preprocessing prior to model training.

Global contrast, expressed as the standard deviation of pixel intensities, is a fundamental indicator of image quality in chest radiography. Diagnostic information in X-ray images depends heavily on subtle grayscale variations that delineate lung fields, soft-tissue boundaries, vascular structures, and osseous anatomy. Low global contrast typically indicates underexposure or suboptimal acquisition, which can obscure anatomical details and reduce the discriminative capacity available to a learning algorithm. Conversely, excessively high global contrast often suggests overprocessing or artificial enhancement, potentially introducing edges and textures not present in standard clinical imaging.

As part of dataset characterization, this metric establishes a baseline reference for expected exposure and contrast levels across the available images. This baseline assists in determining whether contrast normalization methods (e.g., histogram normalization, CLAHE, or no enhancement) are necessary or potentially counterproductive. It also provides a reference against which future inference-time images can be compared, helping to detect contrast outliers that may result from differing acquisition settings, device variability, or prior preprocessing.

In [ ]:
# Compute Global Contrast (standard deviation of pixel intensities)
if scenario("DEVELOPMENT"):

    DF_VIEW["global_std"] = DF_VIEW["image_path"].apply(compute_global_std)

    print("Global contrast computation completed.")
    display(DF_VIEW["global_std"].describe())

In [ ]:
# Plot Lowest, Median and Highest Contrast Images
if scenario("DEVELOPMENT"):

    # Sort DF_VIEW by global contrast
    df_sorted = DF_VIEW.sort_values(by="global_std")

    # Select lowest, median, highest contrast samples
    low_row  = df_sorted.iloc[0]
    mid_row  = df_sorted.iloc[len(df_sorted) // 2]
    high_row = df_sorted.iloc[-1]

    selected = [
        ("Lowest Contrast",  low_row["image_path"],  low_row["global_std"]),
        ("Median Contrast",  mid_row["image_path"],  mid_row["global_std"]),
        ("Highest Contrast", high_row["image_path"], high_row["global_std"])
    ]

    plt.figure(figsize=(16, 7))

    for idx, (label, path, std_val) in enumerate(selected):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Image could not be read: {path}")

        filename = os.path.basename(path)

        plt.subplot(1, 3, idx + 1)
        plt.imshow(img, cmap="gray")
        plt.title(f"{label}\nStd = {std_val:.2f}", fontsize=12)
        plt.axis("off")

        plt.text(
            0.5, -0.10,
            filename,
            fontsize=10,
            ha="center", va="top",
            transform=plt.gca().transAxes
        )

    plt.tight_layout()
    plt.show()

**Figure 2.1** - Lowest, Median and Highest Contrast Images

In [ ]:
# Plot Global Contrast Distribution
if scenario("DEVELOPMENT"):

    plt.figure(figsize=(10, 5))
    plt.hist(DF_VIEW["global_std"].dropna(), bins=40, edgecolor='black')

    plt.title("Global Contrast (Pixel Std) Distribution")
    plt.xlabel("Global Std Value")
    plt.ylabel("Number of Images")
    plt.grid(True)

    plt.show()

**Figure 2.2** - Global Contrast Distribution

The distribution of global contrast values shows a well-formed, approximately Gaussian shape centered around the expected range for standard chest radiographs. This suggests that the majority of images share similar exposure and contrast characteristics, which is desirable for model training. The presence of a small number of low-contrast images at the left tail likely reflects instances of mild underexposure or limited dynamic range. Likewise, the small number of high-contrast cases at the right tail may correspond to images that have undergone stronger post-processing or contrast amplification.

Overall, the dataset appears broadly consistent in terms of contrast quality. The core of the distribution indicates that most images fall within a stable and clinically plausible contrast band, which supports robust model learning. The few outliers observed at the extremes do not dominate the dataset and may be reviewed individually during preprocessing decisions, but they do not suggest any structural imbalance or heterogeneous mixing of incompatible imaging sources.

## 2.2 Dynamic Range

Dynamic range, defined as the difference between the maximum and minimum pixel intensities in an image, serves as a measure of how fully the grayscale space is utilized during acquisition. Chest X-ray imaging relies on a broad representation of intensities to preserve the visibility of both low-attenuation regions such as lung fields and high-attenuation structures such as ribs or the mediastinum. An image with a limited dynamic range may appear washed out or uniformly gray, which reduces the visibility of diagnostically relevant structures. Conversely, an abnormally wide dynamic range may indicate strong post-processing, unusual acquisition parameters, or inconsistent normalization steps applied before dataset construction.

A dataset with large variability in dynamic range can introduce significant inconsistency in the distribution of pixel intensities seen during training. This inconsistency makes it more difficult for a model to learn reliable patterns, as the same anatomical structure may appear substantially lighter or darker depending on acquisition conditions rather than actual physiology. Monitoring this metric makes it possible to identify images that deviate from standard radiographic appearance and that may require normalization or exclusion. A stable and well-defined dynamic range across samples contributes to more predictable model behavior and reduces the risk of learning contrast artifacts.

Dynamic range analysis complements global contrast analysis by providing a direct measurement of the span of intensities present in each image. Whereas the global standard deviation characterizes the distribution’s spread, dynamic range assesses the actual minimum and maximum gray levels used by the imaging system. Together, these metrics assist in determining whether preprocessing steps such as intensity clipping, histogram normalization, or contrast equalization are necessary to achieve a consistent representation of the dataset. They also support the identification of outliers that may result from non-standard imaging devices or prior enhancement.

In [ ]:
# Compute Dynamic Range (max intensity minus min intensity)
if scenario("DEVELOPMENT"):

    # Create the column
    DF_VIEW["dynamic_range"] = DF_VIEW["image_path"].apply(compute_dynamic_range)

    print("Dynamic Range computation completed.")
    display(DF_VIEW["dynamic_range"].describe())

In [ ]:
# Plot distribution of Dynamic Range values
if scenario("DEVELOPMENT"):

    plt.figure(figsize=(10, 5))
    plt.hist(DF_VIEW["dynamic_range"].dropna(), bins=40, edgecolor="black")

    plt.title("Dynamic Range Distribution")
    plt.xlabel("Dynamic Range (Max - Min)")
    plt.ylabel("Number of Images")
    plt.grid(True)

    plt.show()

**Figure 2.3** - Dynamic Range Distribution

Overall, the dataset appears highly uniform in terms of dynamic range. The dominant peak at 255 indicates consistent handling of intensity rescaling across images, which is advantageous for downstream modelling, as the input space remains stable. The small tail of lower dynamic-range images can be inspected individually during the data-quality review phase, but the distribution does not suggest any systemic inconsistency in acquisition or preprocessing.

## 2.3 Histogram Entropy

Histogram entropy measures the complexity and variability of the grayscale distribution in an image. Higher entropy values indicate that pixel intensities are more uniformly distributed across the available range, reflecting greater diversity in tonal values. Lower entropy values occur when intensities are concentrated in a narrower subset of values, producing flatter or more uniform images. In the context of chest radiography, entropy provides an aggregate view of how much tonal information the acquisition contains. Images with low entropy may be underexposed, washed out, or lacking structural definition, while images with very high entropy may have undergone strong contrast equalization or other enhancement steps.

Stable entropy levels across a dataset contribute to consistent feature extraction during training. Variations in entropy reflect differences in perceptual complexity, structural prominence, and contrast distribution, all of which influence how convolutional layers respond to radiographic textures. An image with unusually low entropy may lack sufficient local variation for robust learning, while an image with unusually high entropy may contain contrast artifacts that do not represent normal anatomy. By quantifying entropy across the dataset, it becomes possible to detect samples that deviate from the expected radiographic appearance and that may benefit from targeted normalization or exclusion.

Histogram entropy complements the global standard deviation and dynamic range metrics by capturing the distributional shape of image intensities rather than only the spread or the minimum and maximum values. Together, these metrics help characterize the global exposure profile of the dataset and guide decisions regarding preprocessing steps such as histogram normalization or contrast equalization. Entropy also establishes a reference distribution against which future inference-time inputs can be compared, supporting the identification of atypical acquisition conditions or unexpected preprocessing in production data.

In [ ]:
# Compute Histogram Entropy
if scenario("DEVELOPMENT"):
    DF_VIEW["hist_entropy"] = DF_VIEW["image_path"].apply(compute_hist_entropy)
    display(DF_VIEW["hist_entropy"].describe())

In [ ]:
# Plot representative images for Histogram Entropy
if scenario("DEVELOPMENT"):

    df_sorted = DF_VIEW.sort_values(by="hist_entropy")

    low_row  = df_sorted.iloc[0]
    mid_row  = df_sorted.iloc[len(df_sorted) // 2]
    high_row = df_sorted.iloc[-1]

    examples = [
        ("Lowest Entropy",  low_row["image_path"],  low_row["hist_entropy"]),
        ("Median Entropy",  mid_row["image_path"],  mid_row["hist_entropy"]),
        ("Highest Entropy", high_row["image_path"], high_row["hist_entropy"])
    ]

    plt.figure(figsize=(16, 7))

    for idx, (label, path, value) in enumerate(examples):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        filename = os.path.basename(path)

        plt.subplot(1, 3, idx + 1)
        plt.imshow(img, cmap="gray")
        plt.title(f"{label}\nEntropy = {value:.2f} bits", fontsize=12)
        plt.axis("off")

        plt.text(
            0.5, -0.10, filename,
            fontsize=10,
            ha="center", va="top",
            transform=plt.gca().transAxes
        )

    plt.tight_layout()
    plt.show()

**Figure 2.4** - Lowest, Median, and Highest Entropy Images

In [ ]:
# Plot distribution of Histogram Entropy values
if scenario("DEVELOPMENT"):
    plt.figure(figsize=(10, 5))
    plt.hist(DF_VIEW["hist_entropy"].dropna(), bins=40, edgecolor="black")
    plt.xlabel("Entropy (bits)")
    plt.ylabel("Number of Images")
    plt.grid(True)
    plt.show()

**Figure 2.5** - Histogram Entropy Distribution

The entropy distribution shows a well-defined concentration around values between approximately 6.9 and 7.6 bits, indicating that most images exhibit a balanced spread of pixel intensities across the grayscale range. This range reflects an expected level of tonal complexity for chest radiographs in which both low-attenuation and high-attenuation structures are adequately represented. The presence of a smooth, unimodal shape suggests a consistent image formation process across the dataset, with no major groups of images derived from markedly different acquisition or post-processing pipelines.

A smaller number of images appear in the lower-entropy tail, with values between 6.0 and 6.5 bits. These cases may correspond to underexposed or low-contrast scans in which intensity values are compressed into a narrow portion of the grayscale space. Conversely, the images near the upper end of the distribution reach entropy values close to 7.8 bits, which may indicate stronger histogram equalization or more pronounced tonal variation introduced during preprocessing. Despite these isolated outliers, the overall profile of the entropy distribution reflects a dataset with broadly consistent radiographic complexity, supporting stable feature extraction and learning during model development.

### 2.3.1 Histogram Entropy Outliers

Images with unusually low or high histogram entropy, indicating abnormal pixel-value distributions. Low-entropy examples correspond to radiographs with reduced textural variability or mild posterization, while high-entropy examples reflect excessive noise or aggressive post-processing. These samples lie statistically outside the expected entropy range of the dataset.

In [ ]:
# Detect Histogram Entropy outliers
if scenario("DEVELOPMENT"):
    mean_val = DF_VIEW["hist_entropy"].mean()
    std_val = DF_VIEW["hist_entropy"].std()
    DF_VIEW["hist_entropy_z"] = (DF_VIEW["hist_entropy"] - mean_val) / std_val
    DF_VIEW["hist_entropy_outlier"] = DF_VIEW["hist_entropy_z"].abs() > 3
    
    n_outliers = DF_VIEW["hist_entropy_outlier"].sum()
    pct = (n_outliers / len(DF_VIEW)) * 100
    print(f"Histogram Entropy outliers: {n_outliers} ({pct:.2f}%)")

In [ ]:
# Outlier images for Histogram Entropy
if scenario("DEVELOPMENT"):

    outliers = DF_VIEW[DF_VIEW["hist_entropy_outlier"]]

    if len(outliers) > 0:
        examples = outliers.head(3)

        plt.figure(figsize=(16, 7))

        for idx, (_, row) in enumerate(examples.iterrows()):
            img = cv2.imread(row["image_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            filename = os.path.basename(row["image_path"])

            plt.subplot(1, 3, idx + 1)
            plt.imshow(img, cmap="gray")
            plt.title(f"Outlier\nEntropy = {row['hist_entropy']:.2f}", fontsize=12)
            plt.axis("off")

            plt.text(
                0.5, -0.10, filename,
                fontsize=10,
                ha="center", va="top",
                transform=plt.gca().transAxes
            )

        plt.tight_layout()
        plt.show()

**Figure 2.6** - Histogram Entropy Outliers

## 2.4 Kurtosis and Skewness

Kurtosis and skewness provide statistical descriptions of the shape of the grayscale intensity distribution. Kurtosis quantifies the heaviness of the histogram tails relative to a normal distribution, while skewness measures the asymmetry of the histogram. In chest radiography, the grayscale distribution is typically skewed toward lower intensities due to the predominance of air-filled lung regions. Abnormally low kurtosis may indicate a flatter distribution caused by contrast equalization or smoothing, while unusually high kurtosis suggests a histogram dominated by sharp peaks or extreme intensity clustering.

Atypical kurtosis or skewness values may signal that certain images have undergone nonstandard processing, contain noise patterns, or result from acquisition variations not present in the majority of the dataset. These deviations alter the histogram shape in ways that can mislead convolutional filters during training, especially in the earliest layers where pixel-level distributions influence the learned feature maps. By measuring kurtosis and skewness across the dataset, it becomes possible to detect images whose tonal composition differs substantially from typical chest radiographs and that might warrant further inspection.

Together with global contrast, dynamic range, and entropy, kurtosis and skewness offer a complementary view of image quality by characterizing the higher-order properties of the intensity distribution. These statistics allow the dataset’s overall histogram shape to be compared against expected radiographic norms, supporting the identification of histograms that are unusually flat, unusually peaked, or strongly asymmetric. As part of the data understanding phase, these metrics contribute to determining whether preprocessing steps should normalize intensity distributions or whether the dataset already exhibits stable and consistent characteristics.

In [ ]:
# Compute image shape statistics (kurtosis and skewness)
if scenario("DEVELOPMENT"):
    DF_VIEW["kurtosis"], DF_VIEW["skewness"] = zip(*DF_VIEW["image_path"].apply(compute_kurtosis_skew))
    display(DF_VIEW[["kurtosis", "skewness"]].describe())

In [ ]:
# Plot distribution of kurtosis values
if scenario("DEVELOPMENT"):
    plt.figure(figsize=(10, 5))
    plt.hist(DF_VIEW["kurtosis"].dropna(), bins=40, edgecolor="black")
    plt.xlabel("Kurtosis")
    plt.ylabel("Number of Images")
    plt.grid(True)
    plt.show()

**Figure 2.7** - Kurtosis Distribution

The kurtosis values form a distribution concentrated primarily between approximately -1.5 and +1.0, with a mean near -0.49. This pattern indicates that most images exhibit flatter histograms than a normal distribution would suggest, which is expected for chest radiographs. Air-filled lung fields occupy a substantial portion of the image area and contribute to a large concentration of darker pixel values, while anatomical structures with higher attenuation appear less frequently. A limited number of images show higher kurtosis values extending above 2.0, and a few outliers reach values above 5.0. These cases may correspond to scans with pronounced intensity clustering, possibly caused by strong smoothing, localized contrast amplification, or specific acquisition characteristics. However, these remain isolated and do not indicate systemic inconsistency within the dataset.

In [ ]:
# Plot distribution of skewness values
if scenario("DEVELOPMENT"):
    plt.figure(figsize=(10, 5))
    plt.hist(DF_VIEW["skewness"].dropna(), bins=40, edgecolor="black")
    plt.xlabel("Skewness")
    plt.ylabel("Number of Images")
    plt.grid(True)
    plt.show()

**Figure 2.8** - Skewness Distribution

The skewness distribution is centered around negative values, with a mean near -0.56 and the majority of images falling between approximately -1.2 and -0.3. Negative skewness reflects the expected dominance of lower intensities in chest radiographs, as the lungs represent the largest anatomical region and naturally appear darker relative to bones and mediastinal structures. This asymmetry is characteristic of properly acquired radiographs and confirms that the dataset exhibits the typical tonal bias found in clinical imaging. A small number of images approach or exceed symmetric or positive skewness values, which may indicate unusual exposure conditions, differing preprocessing pipelines, or reduced representation of darker regions. These cases can be reviewed individually but do not appear frequently enough to affect the overall dataset composition.

### 2.4.1 Kurtosis Outliers

Radiographs exhibiting extreme kurtosis values, representing atypical tail behavior in intensity distributions. High-kurtosis images often contain sharp peaks or heavy-tailed noise patterns, while low-kurtosis images show overly flattened histograms. These deviations suggest non-standard imaging characteristics or post-acquisition alterations.

In [ ]:
# Detect Kurtosis outliers
if scenario("DEVELOPMENT"):
    mean_val = DF_VIEW["kurtosis"].mean()
    std_val = DF_VIEW["kurtosis"].std()
    DF_VIEW["kurtosis_z"] = (DF_VIEW["kurtosis"] - mean_val) / std_val
    DF_VIEW["kurtosis_outlier"] = DF_VIEW["kurtosis_z"].abs() > 3
    
    n_outliers = DF_VIEW["kurtosis_outlier"].sum()
    pct = (n_outliers / len(DF_VIEW)) * 100
    print(f"Kurtosis outliers: {n_outliers} ({pct:.2f}%)")

In [ ]:
# Outlier images for Kurtosis
if scenario("DEVELOPMENT"):

    outliers = DF_VIEW[DF_VIEW["kurtosis_outlier"]]

    if len(outliers) > 0:
        examples = outliers.head(3)

        plt.figure(figsize=(16, 7))

        for idx, (_, row) in enumerate(examples.iterrows()):
            img = cv2.imread(row["image_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            filename = os.path.basename(row["image_path"])

            plt.subplot(1, 3, idx + 1)
            plt.imshow(img, cmap="gray")
            plt.title(f"Outlier\nKurtosis = {row['kurtosis']:.2f}", fontsize=12)
            plt.axis("off")

            plt.text(
                0.5, -0.10, filename,
                fontsize=10,
                ha="center", va="top",
                transform=plt.gca().transAxes
            )

        plt.tight_layout()
        plt.show()

**Figure 2.9** - Kurtosis Outliers

## 2.5 Local Contrast Uniformity

Local contrast uniformity describes how contrast varies across different regions of an image. It is computed by dividing the image into smaller tiles and calculating the standard deviation of pixel intensities within each tile. These tile-level measurements reveal how much local structure and intensity variation is present in different parts of the radiograph. Chest X-rays naturally contain heterogeneous regions: lung fields exhibit soft-tissue textures, the mediastinum is denser, and bony structures introduce sharp boundaries. A realistic radiograph therefore displays moderate variation in local contrast between regions.

Significant deviations in local contrast uniformity may indicate that an image has undergone strong local enhancement or smoothing. For example, local histogram equalization methods such as CLAHE (Contrast Limited Adaptive Histogram Equalization) increase local contrast uniformly across the entire image, which reduces the variability between tiles and often raises the average local contrast. Conversely, excessive smoothing lowers local contrast and may disproportionately affect diagnostically important areas. Monitoring this metric helps identify images that display atypical spatial contrast patterns, either due to acquisition characteristics or prior preprocessing, preventing such outliers from influencing the learned feature representations.

Local contrast analysis provides a spatially aware extension of the earlier global metrics. While global contrast and entropy measure aggregation over the entire intensity distribution, local contrast uniformity reveals whether the radiograph preserves appropriate regional differences. This information is especially useful when deciding whether the dataset requires additional normalization steps or whether any images need to be excluded. The metric also establishes a reference for assessing future inference images, helping detect preprocessed or enhanced scans that may not align with the training distribution.

In [ ]:
# Compute Local Contrast Uniformity (mean and variance of tile-level standard deviations)
if scenario("DEVELOPMENT"):
    DF_VIEW["local_std_mean"], DF_VIEW["local_std_var"] = zip(*DF_VIEW["image_path"].apply(compute_local_contrast))
    display(DF_VIEW[["local_std_mean", "local_std_var"]].describe())

In [ ]:
# Plot representative images by local contrast variance
if scenario("DEVELOPMENT"):

    df_sorted = DF_VIEW.sort_values(by="local_std_var")

    low_row  = df_sorted.iloc[0]
    mid_row  = df_sorted.iloc[len(df_sorted) // 2]
    high_row = df_sorted.iloc[-1]

    examples = [
        ("Lowest Local Contrast Variance",  low_row["image_path"],  low_row["local_std_var"]),
        ("Median Local Contrast Variance",  mid_row["image_path"],  mid_row["local_std_var"]),
        ("Highest Local Contrast Variance", high_row["image_path"], high_row["local_std_var"])
    ]

    plt.figure(figsize=(16, 7))

    for idx, (label, path, value) in enumerate(examples):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        filename = os.path.basename(path)

        plt.subplot(1, 3, idx + 1)
        plt.imshow(img, cmap="gray")
        plt.title(f"{label}\nVariance = {value:.2f}", fontsize=12)
        plt.axis("off")

        plt.text(
            0.5, -0.10, filename,
            fontsize=10,
            ha="center", va="top",
            transform=plt.gca().transAxes
        )

    plt.tight_layout()
    plt.show()


**Figure 2.10** - Lowest, Median, and Highest Local Contrast Variance Images

In [ ]:
# Plot distribution of local contrast variance
if scenario("DEVELOPMENT"):
    plt.figure(figsize=(10, 5))
    plt.hist(DF_VIEW["local_std_var"].dropna(), bins=40, edgecolor="black")
    plt.xlabel("Variance of Local Standard Deviations")
    plt.ylabel("Number of Images")
    plt.grid(True)
    plt.show()

**Figure 2.11** - Local Contrast Variance Distribution

The distribution of local contrast variance displays a broad range of values, with most images concentrated between approximately 80 and 300. This behaviour is consistent with chest radiographs, which naturally contain regional differences in contrast: lung fields present relatively homogeneous textures, bony structures introduce strong edges, and the mediastinum contains dense tissue with intermediate variability. The long tail extending toward higher variance suggests that a subset of images exhibits unusually heterogeneous local contrast. These outliers may correspond to radiographs with sharper boundaries, stronger acquisition noise, or more pronounced tonal differences, but they do not form a cluster that would indicate systematic overprocessing or local equalization.

In [ ]:
# Plot distribution of local contrast mean
if scenario("DEVELOPMENT"):
    plt.figure(figsize=(10, 5))
    plt.hist(DF_VIEW["local_std_mean"].dropna(), bins=40, edgecolor="black")
    plt.xlabel("Mean of Local Standard Deviations")
    plt.ylabel("Number of Images")
    plt.grid(True)
    plt.show()

**Figure 2.12** - Local Contrast Mean Distribution

The distribution of mean local contrast shows a dense peak between roughly 20 and 32, reflecting a stable representation of structural detail across the dataset. This profile is characteristic of radiographs that preserve adequate local variation without excessive enhancement. Only a small number of images appear at the lower end of the scale near 11 or at the higher end near 40, which likely represent underexposed scans or images with unusually strong spatial contrast characteristics. These extremes are isolated and do not suggest the presence of a distinct subgroup of preprocessed or contrast-equalized images.

### 2.5.1 Local Contrast Variance Outliers

Images with unusually high or low variance in tile-level standard deviations. High-variance samples typically indicate regions of intensified noise or sharpness inconsistencies, whereas low-variance samples correspond to overly smoothed or low-detail radiographs. These outliers reflect local structural irregularities not representative of the dataset’s general behavior.

In [ ]:
# Detect Local Contrast Variance outliers
if scenario("DEVELOPMENT"):
    mean_val = DF_VIEW["local_std_var"].mean()
    std_val = DF_VIEW["local_std_var"].std()
    DF_VIEW["local_std_var_z"] = (DF_VIEW["local_std_var"] - mean_val) / std_val
    DF_VIEW["local_std_var_outlier"] = DF_VIEW["local_std_var_z"].abs() > 3
    
    n_outliers = DF_VIEW["local_std_var_outlier"].sum()
    pct = (n_outliers / len(DF_VIEW)) * 100
    print(f"Local Contrast Variance outliers: {n_outliers} ({pct:.2f}%)")

In [ ]:
# Outlier images for Local Contrast Variance
if scenario("DEVELOPMENT"):

    outliers = DF_VIEW[DF_VIEW["local_std_var_outlier"]]

    if len(outliers) > 0:
        examples = outliers.head(3)

        plt.figure(figsize=(16, 7))

        for idx, (_, row) in enumerate(examples.iterrows()):
            img = cv2.imread(row["image_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            filename = os.path.basename(row["image_path"])

            plt.subplot(1, 3, idx + 1)
            plt.imshow(img, cmap="gray")
            plt.title(f"Outlier\nVar = {row['local_std_var']:.2f}", fontsize=12)
            plt.axis("off")

            plt.text(
                0.5, -0.10, filename,
                fontsize=10,
                ha="center", va="top",
                transform=plt.gca().transAxes
            )

        plt.tight_layout()
        plt.show()

**Figure 2.13** - Local Contrast Variance Outliers

## 2.6 High-Frequency Content

High-frequency Content (Edge Energy) measures the amount of rapid intensity variation present in an image. It is typically computed by applying a high-pass operator, such as the Laplacian filter, and quantifying the average magnitude of the resulting response. In chest radiography, high-frequency information corresponds to edges and fine structural details such as rib boundaries, vascular markings, clavicle contours, and soft-tissue texture. A well-formed radiograph maintains a balanced level of high-frequency content: excessive smoothing significantly reduces edge energy, while aggressive sharpening or noise amplification increases it.

Large deviations in high-frequency content indicate potential issues in image acquisition or preprocessing. Overly smoothed images may result from denoising pipelines, motion blur, or low-dose acquisition, all of which suppress clinically relevant textural cues. Conversely, images with unusually high edge energy may exhibit strong sharpening, residual noise, or compression artifacts. These variations affect the early convolutional layers of a neural network, which rely heavily on the presence and stability of local gradients to extract meaningful features. By characterizing high-frequency behaviour across the dataset, it becomes possible to identify images whose sharpness profile differs substantially from standard radiographic texture.

High-frequency analysis acts as a complementary measure to the earlier contrast-related metrics by focusing on structural detail rather than tonal distribution. Examining edge energy helps determine whether the dataset contains imaging outliers that may need normalization, exclusion, or preprocessing adjustments. It also establishes a reference for future inference images, enabling the identification of scans that underwent different noise-reduction or sharpening pipelines. Taken together with the preceding metrics, high-frequency content provides a comprehensive overview of the dataset’s radiographic fidelity.

In [ ]:
# Compute high-frequency content using the mean absolute Laplacian response
if scenario("DEVELOPMENT"):
    DF_VIEW["high_freq_energy"] = DF_VIEW["image_path"].apply(compute_high_freq_energy)
    display(DF_VIEW["high_freq_energy"].describe())

In [ ]:
# Plot distribution of high-frequency content values
if scenario("DEVELOPMENT"):
    plt.figure(figsize=(10, 5))
    plt.hist(DF_VIEW["high_freq_energy"].dropna(), bins=40, edgecolor="black")
    plt.xlabel("Mean Absolute Laplacian Value")
    plt.ylabel("Number of Images")
    plt.grid(True)
    plt.show()

**Figure 2.14** - Lowest, Median, and Highest High-Frequency Content Images

In [ ]:
# Plot representative images by high-frequency content
if scenario("DEVELOPMENT"):

    df_sorted = DF_VIEW.sort_values(by="high_freq_energy")

    low_row  = df_sorted.iloc[0]
    mid_row  = df_sorted.iloc[len(df_sorted) // 2]
    high_row = df_sorted.iloc[-1]

    examples = [
        ("Lowest High-Frequency Content",  low_row["image_path"],  low_row["high_freq_energy"]),
        ("Median High-Frequency Content",  mid_row["image_path"],  mid_row["high_freq_energy"]),
        ("Highest High-Frequency Content", high_row["image_path"], high_row["high_freq_energy"])
    ]

    plt.figure(figsize=(16, 7))

    for idx, (label, path, value) in enumerate(examples):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        filename = os.path.basename(path)

        plt.subplot(1, 3, idx + 1)
        plt.imshow(img, cmap="gray")
        plt.title(f"{label}\nHF = {value:.2f}", fontsize=12)
        plt.axis("off")

        plt.text(
            0.5, -0.10, filename,
            fontsize=10,
            ha="center", va="top",
            transform=plt.gca().transAxes
        )

    plt.tight_layout()
    plt.show()

**Figure 2.15** - High-Frequency Content Distribution

The distribution of high-frequency content is concentrated between roughly 3.0 and 7.0, indicating that most images exhibit a normal level of structural detail and edge sharpness consistent with standard chest radiographs. This pattern suggests a uniform acquisition process with stable preservation of fine anatomical boundaries. A smaller number of images fall below this range, which may reflect smoothing, reduced dose, or minor motion effects, while a long upper tail contains isolated cases with unusually strong edges or noise. These outliers remain rare and can be reviewed individually if needed.

Overall, the dataset presents a coherent sharpness profile with only limited deviations, indicating that the radiographs are largely consistent in their high-frequency characteristics and appropriate for further analysis without major preprocessing adjustments.

The image quality metrics collectively indicate that the dataset exhibits a high degree of tonal and structural consistency. Global contrast and dynamic range measurements show that most radiographs maintain appropriate exposure levels, covering nearly the full available grayscale range with limited variation. Histogram entropy reinforces this finding by demonstrating that grayscale complexity remains stable across the dataset, with only a small number of images displaying reduced or unusually elevated entropy.

The distribution of skewness and kurtosis further supports the conclusion that the dataset adheres to typical radiographic intensity profiles. The negative skewness values reflect the expected predominance of darker lung regions, while kurtosis values indicate slightly flattened but otherwise normal histogram shapes. Together, these global metrics suggest that the images have not undergone diverse or aggressive post-processing procedures and that acquisition characteristics are relatively homogeneous.

Spatial metrics derived from local contrast and high-frequency content show comparable stability. Tile-based measures reveal appropriate regional variability in contrast without signs of widespread local enhancement, and the high-frequency content distribution indicates consistent sharpness and preservation of anatomical detail. Only a small number of images deviate noticeably from these patterns, and these cases appear isolated rather than systematic. Overall, the metrics suggest that the dataset is well suited for downstream modelling without the need for substantial corrective preprocessing.

### 2.6.1 High-Frequency Content Outliers

Radiographs with extreme high-frequency responses based on the mean absolute Laplacian. These samples exhibit disproportionate levels of edge enhancement or high-frequency noise, often indicating non-standard sharpening, low-dose acquisition artifacts, or compression effects. Their frequency profiles fall well outside the distribution observed in the dataset.

In [ ]:
# Detect High-Frequency Energy outliers
if scenario("DEVELOPMENT"):
    mean_val = DF_VIEW["high_freq_energy"].mean()
    std_val = DF_VIEW["high_freq_energy"].std()
    DF_VIEW["high_freq_energy_z"] = (DF_VIEW["high_freq_energy"] - mean_val) / std_val
    DF_VIEW["high_freq_energy_outlier"] = DF_VIEW["high_freq_energy_z"].abs() > 3
    
    n_outliers = DF_VIEW["high_freq_energy_outlier"].sum()
    pct = (n_outliers / len(DF_VIEW)) * 100
    print(f"High-Frequency Energy outliers: {n_outliers} ({pct:.2f}%)")

In [ ]:
# Outlier images for High-Frequency Content
if scenario("DEVELOPMENT"):

    outliers = DF_VIEW[DF_VIEW["high_freq_energy_outlier"]]

    if len(outliers) > 0:
        examples = outliers.head(3)

        plt.figure(figsize=(16, 7))

        for idx, (_, row) in enumerate(examples.iterrows()):
            img = cv2.imread(row["image_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            filename = os.path.basename(row["image_path"])

            plt.subplot(1, 3, idx + 1)
            plt.imshow(img, cmap="gray")
            plt.title(f"Outlier\nHF = {row['high_freq_energy']:.2f}", fontsize=12)
            plt.axis("off")

            plt.text(
                0.5, -0.10, filename,
                fontsize=10,
                ha="center", va="top",
                transform=plt.gca().transAxes
            )

        plt.tight_layout()
        plt.show()

**Figure 2.16** - High-Frequency Content Outliers

## 2.7 Geometric Metrics

The preceding contrast and frequency domain metrics characterize tonal and textural properties of the dataset but do not address spatial positioning or image composition. Geometric metrics quantify how X-ray content is positioned and framed within each image, capturing variations in patient centering, field of view, and anatomical coverage. These measurements are essential for deriving data-driven augmentation parameters, particularly for spatial transformations such as translation, zoom, and rotation.

Chest radiographs naturally exhibit variability in patient positioning during acquisition. Even with standardized protocols, factors such as patient cooperation, technician technique, and equipment constraints introduce small deviations in centering and framing. Quantifying these natural geometric variations establishes realistic bounds for augmentation: transformations that exceed the dataset's observed variability risk generating implausible training samples, while transformations that fall short of natural variation fail to improve model robustness.

The computed geometric metrics include horizontal and vertical centering offsets, which measure how far the radiographic content deviates from the image center, and content width and height ratios, which indicate what fraction of the image frame is occupied by actual diagnostic information. Additionally, the aspect ratio of each image is recorded to support quality control and to inform decisions regarding resizing or padding strategies during preprocessing. Together, these metrics enable the derivation of translation and zoom augmentation parameters that reflect the dataset's intrinsic spatial characteristics rather than arbitrary defaults.

In [ ]:
if scenario("DEVELOPMENT"):
    print("Computing geometric metrics...")
    
    (DF_VIEW["centering_offset_x"],
     DF_VIEW["centering_offset_y"],
     DF_VIEW["content_width_ratio"],
     DF_VIEW["content_height_ratio"],
     DF_VIEW["aspect_ratio"]) = zip(*DF_VIEW["image_path"].progress_apply(compute_geometric_metrics))  # ADD aspect_ratio
    
    print("\nGeometric metrics computation completed.")
    display(DF_VIEW[["centering_offset_x", "centering_offset_y", 
                     "content_width_ratio", "content_height_ratio",
                     "aspect_ratio"]].describe())

In [ ]:
# Plot centering offset distributions
if scenario("DEVELOPMENT"):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    axes[0].hist(DF_VIEW["centering_offset_x"].dropna(), bins=40, edgecolor="black")
    axes[0].set_title("Horizontal Centering Offset Distribution")
    axes[0].set_xlabel("Offset (fraction of width)")
    axes[0].set_ylabel("Number of Images")
    axes[0].grid(True)
    axes[0].axvline(0, color='red', linestyle='--', label='Perfect Center')
    axes[0].legend()
    
    axes[1].hist(DF_VIEW["centering_offset_y"].dropna(), bins=40, edgecolor="black")
    axes[1].set_title("Vertical Centering Offset Distribution")
    axes[1].set_xlabel("Offset (fraction of height)")
    axes[1].set_ylabel("Number of Images")
    axes[1].grid(True)
    axes[1].axvline(0, color='red', linestyle='--', label='Perfect Center')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

**Figure 2.17** - Horizontal and Vertical Centering Offset Distribution

The horizontal and vertical centering offset distributions show that most images are well-centered, with offsets concentrated tightly around zero. The standard deviations of 0.5% horizontally and near 0% vertically indicate highly consistent patient positioning across the dataset, suggesting minimal natural variation in image composition.

In [ ]:
# Plot content size ratio distributions
if scenario("DEVELOPMENT"):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    axes[0].hist(DF_VIEW["content_width_ratio"].dropna(), bins=40, edgecolor="black")
    axes[0].set_title("Content Width Ratio Distribution")
    axes[0].set_xlabel("Width Ratio (content / image)")
    axes[0].set_ylabel("Number of Images")
    axes[0].grid(True)
    
    axes[1].hist(DF_VIEW["content_height_ratio"].dropna(), bins=40, edgecolor="black")
    axes[1].set_title("Content Height Ratio Distribution")
    axes[1].set_xlabel("Height Ratio (content / image)")
    axes[1].set_ylabel("Number of Images")
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

**Figure 2.18** - Content Width and Height Ratio Distributions

Content width and height ratios reveal that radiographic content occupies approximately 95-98% of the frame in most images, indicating minimal black borders or empty space. This consistency simplifies preprocessing decisions and suggests that the dataset was acquired with standardized framing protocols.

In [ ]:
# Plot aspect ratio distribution
if scenario("DEVELOPMENT"):
    plt.figure(figsize=(10, 5))
    plt.hist(DF_VIEW["aspect_ratio"].dropna(), bins=40, edgecolor="black")
    plt.title("Aspect Ratio Distribution")
    plt.xlabel("Aspect Ratio (width / height)")
    plt.ylabel("Number of Images")
    plt.grid(True)
    plt.axvline(1.0, color='red', linestyle='--', label='Square (1:1)')
    plt.legend()
    plt.show()

**Figure 2.19** - Aspect Ratio Distribution

The aspect ratio distribution is concentrated around 1.0-1.4, with most images being roughly square or slightly wider than tall. The tight distribution without extreme outliers confirms consistent image acquisition geometry across the dataset, reducing the need for aspect-ratio-specific preprocessing strategies.

## 2.8 Data Understanding Metrics Export

After computing image-level characteristics in the Data Understanding phase, the notebook exports a consolidated JSON file that contains these metrics for every image in the dataset. The file includes the image path, class label, dataset split, all computed quantitative metrics, and the corresponding outlier indicators. It serves as an external representation of the dataset’s quality profile, derived directly from the analytical steps documented in this section.

This exported file is consumed by **_cxray_**, an interactive preprocessing exploration tool developed specifically for this project. The tool is available online at: [https://logus2k.com/cxray](https://logus2k.com/cxray). It provides a visual environment for examining the effects of different preprocessing operations, such as cropping strategies, zoom adjustments, and contrast modifications. When supplied with the exported metrics, the tool allows the user to relate visual inspection to the quantitative properties measured during Data Understanding. This connection supports more informed and consistent Data Preparation decisions, helping ensure that the preprocessing applied during model training is both statistically justified and visually validated.

In [ ]:
# Export DF_VIEW (DU metrics) to JSON for use by the preprocessing UI
if scenario("DEVELOPMENT", "PRODUCTION"):

    export_dir = "./static"
    os.makedirs(export_dir, exist_ok=True)

    export_path = os.path.join(export_dir, "du_metrics.json")

    # Select DU-relevant fields
    DU_EXPORT_COLUMNS = [
        "image_path",
        "label",
        "split",
        "global_std",
        "dynamic_range",
        "hist_entropy",
        "kurtosis",
        "skewness",
        "local_std_mean",
        "local_std_var",
        "high_freq_energy",
        "centering_offset_x",
        "centering_offset_y",
        "content_width_ratio",
        "content_height_ratio",
        "aspect_ratio"
    ]

    # If outlier columns exist, include them automatically
    outlier_cols = [c for c in DF_VIEW.columns if c.endswith("_outlier")]
    DU_EXPORT_COLUMNS.extend(outlier_cols)

    df_export = DF_VIEW[DU_EXPORT_COLUMNS].copy()

    # Normalize file paths for frontend
    df_export["image_path"] = df_export["image_path"].str.replace("\\", "/")

    # Export to JSON
    df_export.to_json(export_path, orient="records", indent=2)

    print(f"DU metrics exported to: {export_path}")

&#160;

# 3. Data Preparation

---

## 3.1 Data Splitting Strategy

The dataset’s predefined train and test partitions are preserved to avoid leakage.
A validation set is created by reserving a fixed proportion of the training data, using a deterministic random seed to ensure reproducibility.
Split sizes and class distributions are documented to support transparent evaluation and to confirm that each subset remains representative of the overall dataset.

In [ ]:
# Isolate the Original Train and Test Data using "split" column
# The Test set is the original data from the TEST_DIR
DF_TEST = DF_MODEL[DF_MODEL['split'] == 'test'].copy()
# The data used for training and validation is the original data from the TRAIN_DIR
DF_TRAIN = DF_MODEL[DF_MODEL['split'] == 'train'].copy()

# Final Test set extraction
X_test = DF_TEST['image_path']
y_test = DF_TEST['label']

# Split the Original Training Data into New Train and Validation Sets
X_train_original = DF_TRAIN['image_path']
y_train_original = DF_TRAIN['label']

# Encode the training labels (0 and 1) for stratification and class weight calculation
y_train_original_encoded = y_train_original.apply(lambda x: 1 if x == 'PNEUMONIA' else 0)

# Split the original training set into the new Training (90%) and Validation (10%) sets
X_train, X_val, y_train, y_val, y_train_encoded, y_val_encoded = train_test_split(
    X_train_original, y_train_original, y_train_original_encoded,
    # 10% of the original training pool is allocated to validation
    test_size=0.10,
    random_state=42, 
    shuffle=True, 
    # Stratify by original label
    stratify=y_train_original
)

# Calculate the class weights using the new training set labels
classes = np.unique(y_train_encoded)
class_weights_array = sk_class_weight.compute_class_weight('balanced', classes=classes, y=y_train_encoded)

# Convert to a dictionary for Keras: {0: weight_for_NORMAL, 1: weight_for_PNEUMONIA}
class_weights_dict = {
    i: class_weights_array[i] for i in range(len(classes))
}

# Variable Summary
print("=" * 70)
print("DATA SPLIT SUMMARY")
print("=" * 70)
print(f"Training set size:   {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size:       {len(X_test)}")
print("-" * 70)
print(f"Final Class Weights (for model.fit): {class_weights_dict}")

# CORRECTED PRINT STATEMENTS USING F-STRINGS:
print(f"   Weight for NORMAL (Class 0): {class_weights_dict[0]:.2f}")
print(f"   Weight for PNEUMONIA (Class 1): {class_weights_dict[1]:.2f}")

print("=" * 70)

## 3.2 Data-Driven Augmentation Parameter Analysis

Data augmentation is a critical component of deep learning pipelines for medical imaging, particularly when working with datasets of limited size or class imbalance. However, the selection of augmentation parameters is often based on ad-hoc experimentation, domain conventions, or arbitrary defaults rather than on systematic analysis of the dataset's inherent characteristics. This approach risks introducing either insufficient variation, failing to improve model generalization, or excessive distortion, creating unrealistic samples that mislead the learning process and degrade performance on genuine clinical data.

The metrics computed in the preceding sections provide a quantitative foundation for deriving augmentation parameters directly from the dataset's natural variation. By measuring the statistical properties of brightness, contrast, spatial positioning, content framing, and noise across the existing images, it becomes possible to establish augmentation ranges that reflect realistic variability rather than speculative guesses. This data-driven approach ensures that augmented samples remain within the distribution of plausible radiographic appearances while still introducing sufficient diversity to prevent overfitting.

Class-specific analysis is particularly important for imbalanced datasets such as this one, where the PNEUMONIA class outnumbers the NORMAL class by a ratio of approximately 2.7:1. Applying uniform augmentation to both classes may not address the representational asymmetry effectively. Instead, examining each class independently allows for tailored augmentation strategies: more aggressive transformation of the minority class to increase its effective sample size, and conservative augmentation of the majority class to preserve its existing diversity without introducing artifacts. This targeted approach supports more balanced learning and reduces the risk of the model developing a systematic bias toward the overrepresented class.

The following analysis computes coefficient of variation (CV) for brightness, contrast, and noise metrics, and standard deviation for geometric positioning and content framing. These statistics are then translated into recommended parameter ranges for Keras preprocessing layers (`RandomRotation`, `RandomTranslation`, `RandomZoom`, `RandomBrightness`, `RandomContrast`, and `GaussianNoise`), providing concrete, defensible values that can be directly applied during model training through a `Sequential` augmentation pipeline. The analysis is performed separately for NORMAL and PNEUMONIA images, and concludes with strategic recommendations for handling class imbalance through either differential augmentation or class weighting.

In [ ]:
if scenario("DEVELOPMENT"):
    print("=" * 70)
    print("DATA-DRIVEN AUGMENTATION PARAMETER ANALYSIS (PER CLASS)")
    print("=" * 70)
    print()
    
    for class_name in ["NORMAL", "PNEUMONIA"]:
        df_class = DF_VIEW[DF_VIEW["label"] == class_name]
        n_samples = len(df_class)
        
        print(f"\n{'='*70}")
        print(f"CLASS: {class_name} ({n_samples} images)")
        print(f"{'='*70}\n")
        
        # ===== BRIGHTNESS/CONTRAST AUGMENTATION =====
        global_stds = df_class["global_std"].dropna()
        # CV calculation is correct: std / mean
        brightness_cv = global_stds.std() / global_stds.mean()
        
        # This becomes the factor for RandomBrightness, capped at 0.5
        brightness_var_recommended = min(0.5, brightness_cv)
        
        # This old-style range is informative but not directly used in modern Keras factor
        brightness_range_old_style = (max(0.5, 1 - brightness_var_recommended), 
                                      min(1.5, 1 + brightness_var_recommended))
        
        print("BRIGHTNESS AUGMENTATION:")
        print(f"  Natural variation (CV): {brightness_cv:.2%}")
        # Show recommended factor for RandomBrightness
        print(f"  Recommended brightness_factor: {brightness_var_recommended:.2f}")
        print(f"  (Approximate old-style range: {brightness_range_old_style})")
        print()
        
        # ===== CONTRAST AUGMENTATION =====
        hist_entropies = df_class["hist_entropy"].dropna()
        contrast_cv = hist_entropies.std() / hist_entropies.mean()
        
        # This becomes the factor for RandomContrast
        contrast_var_recommended = min(0.5, contrast_cv)
        
        print("CONTRAST AUGMENTATION:")
        print(f"  Natural variation (CV): {contrast_cv:.2%}")
        print(f"  Recommended contrast_factor: {contrast_var_recommended:.2f}")
        print()
        
        # ===== GEOMETRIC AUGMENTATION =====
        offset_x_std = df_class["centering_offset_x"].dropna().std()
        offset_y_std = df_class["centering_offset_y"].dropna().std()
        
        width_ratio_std = df_class["content_width_ratio"].dropna().std()
        height_ratio_std = df_class["content_height_ratio"].dropna().std()
        
        # This becomes the height_factor for RandomZoom (symmetric zoom)
        zoom_var_recommended = max(width_ratio_std, height_ratio_std)
        
        print("GEOMETRIC AUGMENTATION:")
        print(f"  Horizontal centering std: {offset_x_std:.3f}")
        print(f"  Vertical centering std: {offset_y_std:.3f}")
        # Show recommended factors for RandomTranslation
        print(f"  Recommended width_factor (Translation): {offset_x_std:.3f}")
        print(f"  Recommended height_factor (Translation): {offset_y_std:.3f}")
        # Show recommended factor for RandomZoom
        print(f"  Recommended zoom_height_factor: {zoom_var_recommended:.3f}")
        print()
        
        # ===== NOISE AUGMENTATION =====
        high_freq = df_class["high_freq_energy"].dropna()
        noise_cv = high_freq.std() / high_freq.mean()

        if noise_cv > 0.3:
            gaussian_noise_recommended = min(0.05, noise_cv * 0.1)  # Changed: 0.05 cap, 0.1 scale
        else:
            gaussian_noise_recommended = 0
            print("NOISE AUGMENTATION:")
            print(f"  Recommended gaussian_noise (stddev): 0 (low variation)")
        print()
        
        # ===== CLASS-SPECIFIC RECOMMENDATIONS (MODERN KERAS API) =====
        print("-" * 70)
        print(f"RECOMMENDED CONFIGURATION FOR {class_name} (Modern Keras Layers):")
        print("-" * 70)
        # Use factor (fraction of 2*PI, or 1/360th of the degrees)
        print(f"RandomRotation(factor={10/360.0:.3f}),  # 10 degrees") 
        # width_factor and height_factor for RandomTranslation
        print(f"RandomTranslation(width_factor={offset_x_std:.3f}, height_factor={offset_y_std:.3f}),")
        # factor for RandomBrightness
        print(f"RandomBrightness(factor={brightness_var_recommended:.2f}),") 
        # height_factor for RandomZoom (symmetric zoom)
        print(f"RandomZoom(height_factor={zoom_var_recommended:.3f}),")
        print(f"GaussianNoise(stddev={gaussian_noise_recommended:.4f})")
        print()
    
    # ===== BALANCING STRATEGY =====
    print("\n" + "=" * 70)
    print("CLASS IMBALANCE HANDLING STRATEGY")
    print("=" * 70)
    print()
    
    normal_count = len(DF_VIEW[DF_VIEW["label"] == "NORMAL"])
    pneumonia_count = len(DF_VIEW[DF_VIEW["label"] == "PNEUMONIA"])
    imbalance_ratio = pneumonia_count / normal_count
    
    print(f"NORMAL: {normal_count} images")
    print(f"PNEUMONIA: {pneumonia_count} images")
    print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")
    print()
    
    print("RECOMMENDED APPROACH:")
    print()
    
    # --- Corrected Option 1: Separate Keras Sequential Models ---
    print("Option 1: Separate Keras Sequential Augmentation (More Aggressive for NORMAL)")
    print("-" * 70)
    print("# More aggressive augmentation for minority class (NORMAL)")
    print("normal_augmentation_pipeline = Sequential([")
    print("    # Rotation: More than natural 15 degrees (~0.042 factor)")
    print("    RandomRotation(factor=0.042, fill_mode='constant', fill_value=0.0),") 
    print("    # Translation: More aggressive (0.10 factor)")
    print("    RandomTranslation(height_factor=0.10, width_factor=0.10, fill_mode='constant'),") 
    print("    # Brightness: More aggressive (0.20 factor)")
    print("    RandomBrightness(factor=0.20, value_range=(0, 1)),") 
    print("    # Zoom: More aggressive (0.10 factor)")
    print("    RandomZoom(height_factor=0.10, fill_mode='constant', fill_value=0.0),")
    print("    RandomFlip(mode='horizontal')")
    print("])")
    print()
    
    print("# Conservative augmentation for majority class (PNEUMONIA)")
    print("pneumonia_augmentation_pipeline = Sequential([")
    print("    # Rotation: Moderate (10 degrees ~0.028 factor)")
    print("    RandomRotation(factor=0.028, fill_mode='constant', fill_value=0.0),") 
    print("    # Translation: Match natural variation (~0.005 factor)")
    print("    RandomTranslation(height_factor=0.005, width_factor=0.005, fill_mode='constant'),")
    print("    # Brightness: Moderate (0.15 factor)")
    print("    RandomBrightness(factor=0.15, value_range=(0, 1))")
    print("])")
    print()
    
    # --- Corrected Option 2: Unified Generator + Class Weights ---
    print("Option 2: Unified Keras Sequential Augmentation + Class Weights (Recommended)")
    print("-" * 70)
    print("# Single pipeline with moderate augmentation")
    print("unified_augmentation_pipeline = Sequential([")
    print("    RandomRotation(factor=0.028),  # 10 degrees")
    print("    RandomTranslation(width_factor=0.05, height_factor=0.05),")
    print("    RandomBrightness(factor=0.15, value_range=(0, 1)),")
    print("    RandomFlip(mode='horizontal')")
    print("])")
    print()
    
    print("# Use class weights to handle imbalance (applied in model.fit)")
    print("class_weights = sk_class_weight.compute_class_weight(")
    print("    'balanced',")
    print("    classes=np.unique(y_train),")
    print("    y=y_train")
    print(")")
    print(f"# Estimated weights: NORMAL={imbalance_ratio:.2f}, PNEUMONIA=1.0")

## 3.2 Data Augmentation

Augmentation is applied exclusively to the training set and kept consistent across all models to enable fair comparison.
Transformations are limited to clinically realistic variations that simulate routine acquisition differences, such as small rotations, translations, mild zoom, and conservative brightness adjustments.
Anatomically implausible transformations, including vertical flips or aggressive geometric distortions, are avoided.
Augmentation ranges are selected to preserve diagnostic content while improving generalization under typical real-world imaging variability.

In [ ]:
if scenario("DEVELOPMENT"):
    print("=" * 70)
    print("DATA-DRIVEN AUGMENTATION PARAMETER ANALYSIS AND VARIABLE INITIALIZATION")
    print("=" * 70)

    # 1. INITIALIZE DICTIONARIES TO STORE CLASS-SPECIFIC RESULTS
    class_results = {}
    
    # Static parameters are initialized here
    # 10 degrees
    CALCULATED_ROTATION_FACTOR = 10 / 360.0
    # Only horizontal makes sense in this case
    CALCULATED_FLIP_MODE = 'horizontal'
    
    # 2. CALCULATE AND STORE CLASS-SPECIFIC VALUES
    for class_name in ["NORMAL", "PNEUMONIA"]:

        df_class = DF_VIEW[DF_VIEW["label"] == class_name]
        
        # BRIGHTNESS/CONTRAST
        global_stds = df_class["global_std"].dropna()
        brightness_cv = global_stds.std() / global_stds.mean()
        brightness_var_recommended = min(0.5, brightness_cv)
        
        hist_entropies = df_class["hist_entropy"].dropna()
        contrast_cv = hist_entropies.std() / hist_entropies.mean()
        contrast_var_recommended = min(0.5, contrast_cv)
        
        # GEOMETRIC
        offset_x_std = df_class["centering_offset_x"].dropna().std()
        offset_y_std = df_class["centering_offset_y"].dropna().std()
        width_ratio_std = df_class["content_width_ratio"].dropna().std()
        height_ratio_std = df_class["content_height_ratio"].dropna().std()
        zoom_var_recommended = max(width_ratio_std, height_ratio_std)
        
        # NOISE - More conservative calculation
        high_freq = df_class["high_freq_energy"].dropna()
        noise_cv = high_freq.std() / high_freq.mean()

        if noise_cv > 0.5:  # Only if variation is very high
            gaussian_noise_recommended = min(0.05, noise_cv * 0.1)  # Cap at 5%, scale by 0.1
        else:
            gaussian_noise_recommended = 0
            
        # Store results
        class_results[class_name] = {
            'brightness_factor': brightness_var_recommended,
            'contrast_factor': contrast_var_recommended,
            'offset_x_std': offset_x_std,
            'offset_y_std': offset_y_std,
            'zoom_factor': zoom_var_recommended,
            'gaussian_stddev': gaussian_noise_recommended,
            'count': len(df_class)
        }
        
        # Printing for development environment confirmation
        print(f"\n--- {class_name} Parameters ---")
        print(f"Brightness Factor: {brightness_var_recommended:.3f}")
        print(f"Contrast Factor: {contrast_var_recommended:.3f}")
        print(f"Shift X/Y Std: {offset_x_std:.3f} / {offset_y_std:.3f}")
        print(f"Zoom Factor: {zoom_var_recommended:.3f}")
        print(f"Gaussian Noise: {gaussian_noise_recommended:.3f}")

    # 3. ARTIFACT DETECTION: Calculate class difference ratios
    print("\n" + "=" * 70)
    print("ARTIFACT DETECTION: Analyzing Class Distribution Differences")
    print("=" * 70)
    
    def safe_ratio(a, b, epsilon=0.001):
        """Calculate ratio, avoiding division by zero"""
        return max(a, b) / max(min(a, b), epsilon)
    
    brightness_ratio = safe_ratio(
        class_results['NORMAL']['brightness_factor'],
        class_results['PNEUMONIA']['brightness_factor']
    )
    
    contrast_ratio = safe_ratio(
        class_results['NORMAL']['contrast_factor'],
        class_results['PNEUMONIA']['contrast_factor']
    )
    
    offset_x_ratio = safe_ratio(
        class_results['NORMAL']['offset_x_std'],
        class_results['PNEUMONIA']['offset_x_std']
    )
    
    offset_y_ratio = safe_ratio(
        class_results['NORMAL']['offset_y_std'],
        class_results['PNEUMONIA']['offset_y_std']
    )
    
    zoom_ratio = safe_ratio(
        class_results['NORMAL']['zoom_factor'],
        class_results['PNEUMONIA']['zoom_factor']
    )
    
    print(f"Brightness variation ratio: {brightness_ratio:.2f}x")
    print(f"Contrast variation ratio: {contrast_ratio:.2f}x")
    print(f"Horizontal centering ratio: {offset_x_ratio:.2f}x")
    print(f"Vertical centering ratio: {offset_y_ratio:.2f}x")
    print(f"Zoom/framing ratio: {zoom_ratio:.2f}x")
    
    # 4. ENHANCED REGULARIZATION: Boost augmentation where classes differ
    print("\n" + "=" * 70)
    print("REGULARIZATION STRATEGY: Artifact-Aware Augmentation")
    print("=" * 70)
    
    # BRIGHTNESS: If classes differ significantly, boost augmentation
    base_brightness = max(
        class_results['NORMAL']['brightness_factor'], 
        class_results['PNEUMONIA']['brightness_factor']
    )
    
    if brightness_ratio > 1.5:
        CALCULATED_BRIGHTNESS_FACTOR = min(0.3, base_brightness * 1.2)
        print(f"⚠️  Brightness: {brightness_ratio:.1f}x class difference detected")
        print(f"   → Boosting from {base_brightness:.3f} to {CALCULATED_BRIGHTNESS_FACTOR:.3f} (20% increase)")
    else:
        CALCULATED_BRIGHTNESS_FACTOR = base_brightness
        print(f"✅ Brightness: No significant class difference ({brightness_ratio:.1f}x)")
        print(f"   → Using base value: {CALCULATED_BRIGHTNESS_FACTOR:.3f}")
    
    # CONTRAST: If classes differ significantly, boost augmentation
    base_contrast = max(
        class_results['NORMAL']['contrast_factor'], 
        class_results['PNEUMONIA']['contrast_factor']
    )
    
    if contrast_ratio > 1.5:
        CALCULATED_CONTRAST_FACTOR = min(0.3, base_contrast * 1.2)
        print(f"⚠️  Contrast: {contrast_ratio:.1f}x class difference detected")
        print(f"   → Boosting from {base_contrast:.3f} to {CALCULATED_CONTRAST_FACTOR:.3f} (20% increase)")
    else:
        CALCULATED_CONTRAST_FACTOR = base_contrast
        print(f"✅ Contrast: No significant class difference ({contrast_ratio:.1f}x)")
        print(f"   → Using base value: {CALCULATED_CONTRAST_FACTOR:.3f}")
    
    # TRANSLATION: If centering differs significantly, force stronger augmentation
    base_offset_x = max(
        class_results['NORMAL']['offset_x_std'],
        class_results['PNEUMONIA']['offset_x_std']
    )
    base_offset_y = max(
        class_results['NORMAL']['offset_y_std'],
        class_results['PNEUMONIA']['offset_y_std']
    )
    base_translation = max(base_offset_x, base_offset_y)
    
    if offset_x_ratio > 2.0:
        CALCULATED_WIDTH_FACTOR = max(base_translation, 0.05)
        print(f"⚠️  Horizontal centering: {offset_x_ratio:.1f}x class difference detected")
        print(f"   → Forcing minimum 5% translation (was {base_translation:.3f})")
        print(f"   → This destroys spurious centering correlation")
    elif offset_x_ratio > 1.5:
        CALCULATED_WIDTH_FACTOR = max(base_translation, 0.03)
        print(f"⚠️  Horizontal centering: {offset_x_ratio:.1f}x class difference detected")
        print(f"   → Boosting to minimum 3% translation (was {base_translation:.3f})")
    else:
        CALCULATED_WIDTH_FACTOR = base_translation
        print(f"✅ Horizontal centering: No significant class difference ({offset_x_ratio:.1f}x)")
        print(f"   → Using base value: {CALCULATED_WIDTH_FACTOR:.3f}")
    
    # ZOOM: If framing differs significantly, boost augmentation
    base_zoom = max(
        class_results['NORMAL']['zoom_factor'],
        class_results['PNEUMONIA']['zoom_factor']
    )
    
    if zoom_ratio > 1.5:
        CALCULATED_ZOOM_FACTOR = min(0.15, base_zoom * 1.3)
        print(f"⚠️  Zoom/framing: {zoom_ratio:.1f}x class difference detected")
        print(f"   → Boosting from {base_zoom:.3f} to {CALCULATED_ZOOM_FACTOR:.3f} (30% increase)")
    else:
        CALCULATED_ZOOM_FACTOR = base_zoom
        print(f"✅ Zoom/framing: No significant class difference ({zoom_ratio:.1f}x)")
        print(f"   → Using base value: {CALCULATED_ZOOM_FACTOR:.3f}")
    
    # NOISE: Use conservative approach (cap at 5%)
    CALCULATED_GAUSSIAN_STDDEV = max(
        class_results['NORMAL']['gaussian_stddev'], 
        class_results['PNEUMONIA']['gaussian_stddev']
    )
    
    if CALCULATED_GAUSSIAN_STDDEV > 0:
        print(f"⚠️  Noise: Detected sensor noise variation")
        print(f"   → Adding conservative {CALCULATED_GAUSSIAN_STDDEV:.3f} Gaussian noise")
    else:
        print(f"✅ Noise: No significant noise variation detected")
        print(f"   → No noise augmentation needed")

    # 5. CALCULATE CLASS WEIGHTS FOR IMBALANCE HANDLING
    normal_count = class_results['NORMAL']['count']
    pneumonia_count = class_results['PNEUMONIA']['count']
    imbalance_ratio = pneumonia_count / normal_count
    
    # Store the required counts
    NORMAL_COUNT = normal_count
    PNEUMONIA_COUNT = pneumonia_count
    
    print("\n" + "=" * 70)
    print("FINAL TRAINING VARIABLES INITIALIZED:")
    print("=" * 70)
    
    print(f"\nAUGMENTATION PARAMETERS:")
    print(f"CALCULATED_ROTATION_FACTOR     = {CALCULATED_ROTATION_FACTOR:.4f} (10°)")
    print(f"CALCULATED_WIDTH_FACTOR        = {CALCULATED_WIDTH_FACTOR:.4f}")
    print(f"CALCULATED_ZOOM_FACTOR         = {CALCULATED_ZOOM_FACTOR:.4f}")
    print(f"CALCULATED_BRIGHTNESS_FACTOR   = {CALCULATED_BRIGHTNESS_FACTOR:.4f}")
    print(f"CALCULATED_CONTRAST_FACTOR     = {CALCULATED_CONTRAST_FACTOR:.4f}")
    print(f"CALCULATED_GAUSSIAN_STDDEV     = {CALCULATED_GAUSSIAN_STDDEV:.4f}")
    print(f"CALCULATED_FLIP_MODE           = '{CALCULATED_FLIP_MODE}'")
    
    print(f"\nCLASS BALANCE:")
    print(f"NORMAL_COUNT    = {NORMAL_COUNT}")
    print(f"PNEUMONIA_COUNT = {PNEUMONIA_COUNT}")
    print(f"Imbalance Ratio = {imbalance_ratio:.2f}:1 (PNEUMONIA:NORMAL)")

In [ ]:
CALCULATED_ROTATION_FACTOR     = 5 / 360
CALCULATED_WIDTH_FACTOR        = 0.05
CALCULATED_ZOOM_FACTOR         = 0.1
CALCULATED_BRIGHTNESS_FACTOR   = 0.25
CALCULATED_CONTRAST_FACTOR     = 0.05
CALCULATED_GAUSSIAN_STDDEV     = 0.02
CALCULATED_FLIP_MODE           = 'horizontal'


# Use the variables initialized in DATA-DRIVEN AUGMENTATION PARAMETER ANALYSIS
unified_augmentation_pipeline = Sequential([
    RandomRotation(factor=CALCULATED_ROTATION_FACTOR),
    RandomTranslation(
        # Using same factor for both (aggregated from max of all offsets)
        height_factor=CALCULATED_WIDTH_FACTOR,
        width_factor=CALCULATED_WIDTH_FACTOR,
        fill_mode='constant',
        fill_value=0.0
    ),
    RandomZoom(height_factor=CALCULATED_ZOOM_FACTOR),
    RandomFlip(mode=CALCULATED_FLIP_MODE),
    RandomBrightness(factor=CALCULATED_BRIGHTNESS_FACTOR, value_range=(0, 1)),
    RandomContrast(factor=CALCULATED_CONTRAST_FACTOR, value_range=(0, 1)),
    GaussianNoise(stddev=CALCULATED_GAUSSIAN_STDDEV)
])

## 3.3 Model-Specific Preprocessing Pipelines

Each model architecture employs a defined preprocessing pipeline, including resizing, channel arrangement, and normalization.
Augmentation is applied before model-specific normalization to avoid altering standardized distributions.
Pipelines for convolutional networks replicate grayscale images into three channels and apply ImageNet normalization when pretrained backbones are used.
Transformers follow similar pipelines with model-appropriate input resolutions.
All steps are implemented within tf.data for efficiency and consistency.

In [ ]:
# Define global constants consistent with each model expected input
IMG_SIZE = 256
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

In [ ]:
def load_and_preprocess_image(filepath, label):
    """Loads image, resizes, converts to grayscale, and normalizes."""
    # Load image file
    img = tf.io.read_file(filepath)
    
    # Decode to tensor (3 channels is a safe default for decoding)
    img = tf.image.decode_jpeg(img, channels=3)
    
    # Resize image to model input size
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    
    # Convert to grayscale (1 channel)
    img = tf.image.rgb_to_grayscale(img)
    
    # Normalize to [0, 1] range (CRITICAL for RandomBrightness/Contrast layers)
    img = img / 255.0
    
    # Convert label to float if using binary_crossentropy/sigmoid output
    label = tf.cast(label == 'PNEUMONIA', tf.float32)
    
    return img, label

In [ ]:
# Training Dataset (augmentated)
TRAIN_DS = tf.data.Dataset.from_tensor_slices((X_train, y_train))
TRAIN_DS = TRAIN_DS.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

# Use the Data-Driven Augmentation Pipeline
TRAIN_DS = TRAIN_DS.map(
    lambda x, y: (unified_augmentation_pipeline(x), y), 
    num_parallel_calls=tf.data.AUTOTUNE
)

# Optimization chain (batching and prefetching)
TRAIN_DS = (
    TRAIN_DS
    .cache()             
    .shuffle(len(X_train)) 
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE) 
)

# Validation dataset (not augmentated)
VAL_DS = tf.data.Dataset.from_tensor_slices((X_val, y_val))
VAL_DS = VAL_DS.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

# Optimization chain
VAL_DS = (
    VAL_DS
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Test dataset (not augmentated)
TEST_DS = tf.data.Dataset.from_tensor_slices((X_test, y_test))
TEST_DS = TEST_DS.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

# Optimization chain
TEST_DS = (
    TEST_DS
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

## 3.4 Dataset Pipeline Validation

In [ ]:
print("\n" + "=" * 70)
print("DATASET PIPELINE VALIDATION")
print("=" * 70)

# Check Training Dataset
for images, labels in TRAIN_DS.take(1): # type: ignore
    print("\n--- Training Dataset (TRAIN_DS) ---")
    print(f"Batch Shape: {images.shape} (Expected: [{BATCH_SIZE}, {IMG_SIZE}, {IMG_SIZE}, 1])")
    print(f"Labels Shape: {labels.shape} (Expected: [{BATCH_SIZE},])")
    print(f"Pixel Range: [{tf.reduce_min(images):.3f}, {tf.reduce_max(images):.3f}] (Should be close to [0.0, 1.0])")
    print(f"Labels (First 5): {labels.numpy()[:5]}")
    break

# Check Validation Dataset
for images, labels in VAL_DS.take(1): # type: ignore
    print("\n--- Validation Dataset (VAL_DS) ---")
    print(f"Batch Shape: {images.shape} (Expected: [{BATCH_SIZE}, {IMG_SIZE}, {IMG_SIZE}, 1])")
    print(f"Labels Shape: {labels.shape} (Expected: [{BATCH_SIZE},])")
    print(f"Pixel Range: [{tf.reduce_min(images):.3f}, {tf.reduce_max(images):.3f}] (Should be [0.0, 1.0])")
    print(f"Labels (First 5): {labels.numpy()[:5]}")
    break

# Check Test Dataset
for images, labels in TEST_DS.take(1): # type: ignore
    print("\n--- Test Dataset (TEST_DS) ---")
    print(f"Batch Shape: {images.shape} (Expected: [{BATCH_SIZE}, {IMG_SIZE}, {IMG_SIZE}, 1])")
    print(f"Labels Shape: {labels.shape} (Expected: [{BATCH_SIZE},])")
    print(f"Pixel Range: [{tf.reduce_min(images):.3f}, {tf.reduce_max(images):.3f}] (Should be [0.0, 1.0])")
    print(f"Labels (First 5): {labels.numpy()[:5]}")
    break

&#160;

# 4. Modeling

---

## 4.1 Model Checkpoint Saving

We set up a callback to save the best model during training based on validation PR-AUC rather than raw accuracy. This is more appropriate for an imbalanced, clinically oriented problem where the balance between precision and recall is more important than overall accuracy.

The ModelCheckpoint callback:

* Saves the model to the file path defined by `CHECKPOINT_PATH`.
* Monitors the `val_pr_auc` metric on the validation set.
* Keeps only the model corresponding to the highest observed `val_pr_auc` during training.

By saving the best validation-PR-AUC checkpoint instead of simply using the weights from the final epoch, we reduce the risk of overfitting. If performance on the validation set starts to degrade after a certain point, we still retain the version of the model that showed the best generalization according to the chosen metric.

In [ ]:
# Model checkpoint saving
best_model_callback = ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    monitor="val_pr_auc",
    mode="max",
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

## 4.2 Early Stopping

We also use an EarlyStopping callback to automatically stop training when further epochs no longer improve the model’s performance on the validation set.

In this project:

* EarlyStopping monitors the validation PR-AUC (`val_pr_auc`), not the training metrics.
* If `val_pr_auc` does not improve for a specified number of epochs (the patience value), training is stopped.
* With `restore_best_weights=True`, the model weights are rolled back to those from the epoch with the highest `val_pr_auc`.

This prevents wasting epochs once the model has effectively converged and reduces overfitting by avoiding continued training after validation performance has stopped improving, while still keeping the best-performing version of the model for subsequent evaluation.

In [ ]:
# Early stopping
early_stopping = EarlyStopping(
    monitor="val_pr_auc",
    mode="max",
    # Generous patience because training with AdamW and LR scheduling can be slow
    patience=15,
    restore_best_weights=True,
    verbose=1
)

## 4.3 Learning Rate

The learning rate is a key hyperparameter that controls how much the model weights are updated during training. If it is too high, the optimizer may overshoot good minima and converge to a suboptimal solution; if it is too low, training can become very slow or get stuck.

In this project, we use the AdamW optimizer with an initial learning rate defined by `LEARNING_RATE`, together with a learning rate scheduler based on the `ReduceLROnPlateau` callback. The scheduler:

* Monitors the validation loss (`val_loss`).
* If `val_loss` does not improve for a certain number of epochs (patience), it reduces the learning rate by a given factor.
* Ensures that the learning rate never goes below a specified minimum (`min_lr`).

This strategy allows the model to make relatively large updates in the early stages of training, then gradually switch to smaller, finer updates once progress slows. In practice, this helps the model converge more reliably and can improve final validation performance without manual tuning of the learning rate schedule.

In [ ]:
# Learning rate scheduler
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    mode="min",
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

## 4.4 Build

In [ ]:
# EfficientNetV2-B0 Model Definition

# Input Layer (256×256×1 grayscale)
input_tensor = Input(shape=(256, 256, 1))

# Grayscale to RGB (without using a lambda)
x = Concatenate(axis=-1)([input_tensor, input_tensor, input_tensor])

# Load EfficientNetV2-B0 backbone
base_model = EfficientNetV2B0(
    include_top=False,
    weights="imagenet",
    input_tensor=x
)

# 4. Freeze backbone
base_model.trainable = False

# 5. Classification Head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(
    128,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-4)
)(x)
x = Dropout(0.4)(x)
output = Dense(1, activation="sigmoid")(x)

# 6. Assemble model
model = Model(inputs=input_tensor, outputs=output)

model.summary()

## 4.5 Two-Stage Training: Feature Extraction and Fine-Tuning

To leverage the power of pre-trained models like EfficientNet, we employ a **Two-Stage Transfer Learning** strategy. This approach maximizes performance while protecting the knowledge learned from the massive ImageNet dataset.

1.  **Stage 1: Feature Extraction (Base Frozen):** The pre-trained convolutional base (EfficientNet) is **frozen** to lock its weights. Only the newly added classification head is trained. This "warms up" the dense layers to interpret the base's features using a standard learning rate.
2.  **Stage 2: Fine-Tuning (Base Unfrozen):** After the head is stable, the entire model is **unfrozen**, but the learning rate is dramatically reduced ($\mathbf{10^{-5}}$). This allows the base model to slightly adapt its features specifically to the texture and anatomy of X-ray images, ensuring stable convergence and optimal results.

### 4.5.1 Feature Extraction (Base Frozen)

In this initial stage, the base model (`EfficientNetV2B0` or `ResNet50V2`) is set to **un-trainable**. We compile the model and train it for a small number of epochs (15 in this case) using a standard learning rate ($\mathbf{10^{-3}}$).

The primary goal is to **train the new classification head** (GlobalAveragePooling + Dense layers) to correctly map the frozen features produced by the base to our specific binary labels (Normal/Pneumonia). This stabilizes the head before we attempt fine-tuning.

In [ ]:
print("--- STAGE 1: Feature Extraction (Base Frozen) ---")

model.compile(
    optimizer=AdamW(learning_rate=1e-3, weight_decay=1e-4),   # type: ignore
    loss="binary_crossentropy",
    metrics=["accuracy", AUC(name="auc"), AUC(curve="PR", name="pr_auc"), Recall(name="recall")]
)

history_stage1 = model.fit(
    TRAIN_DS,
    validation_data=VAL_DS,
    epochs=15, # Shorter run
    callbacks=[best_model_callback, early_stopping], # Reuse callbacks
    class_weight=class_weights_dict
)

### 4.5.2 Fine-Tuning (Base Unfrozen)

After Stage 1 stabilizes the classification head, we proceed to **unfreeze the entire base model** (`base_model.trainable = True`).

To prevent the destruction of the pre-trained weights, the model is **recompiled** with a **significantly lower learning rate** ($\mathbf{10^{-5}}$). This low rate ensures the training process only makes small, careful adjustments to the base model's weights, adapting them to the X-ray domain while maintaining a focus on high validation PR-AUC. The learning rate scheduler and early stopping continue to manage the process.

In [ ]:
print("\n--- STAGE 2: Fine-Tuning (Base Unfrozen) ---")

# Unfreeze the base model
base_model.trainable = True

# Optional: Fine-tune only the top N layers if full unfreeze is too unstable
# for layer in base_model.layers[:-50]:
#    layer.trainable = False

# Recompile with a VERY LOW learning rate (Crucial!)
model.compile(
    # 10x or 100x smaller LR
    optimizer=AdamW(learning_rate=1e-5, weight_decay=1e-4), # type: ignore
    loss="binary_crossentropy",
    metrics=["accuracy", AUC(name="auc"), AUC(curve="PR", name="pr_auc"), Recall(name="recall")]
)

# Continue training
# Note: initial_epoch ensures the plot continuity
total_epochs = 100
history_fine = model.fit(
    TRAIN_DS,
    validation_data=VAL_DS,
    epochs=total_epochs,
    initial_epoch=history_stage1.epoch[-1], 
    callbacks=[best_model_callback, early_stopping, lr_scheduler],
    class_weight=class_weights_dict
)

&#160;

# 5. Evaluation

---

In this section, the final model is evaluated on a held-out test set using clinically relevant classification metrics (accuracy, precision, recall, F1-score and AUC), together with confusion matrices, Precision–Recall analysis and threshold-based error summaries. We also inspect the loss and accuracy curves over epochs to verify that training converged without severe overfitting before selecting the best checkpoint for testing.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Combine History Data ---

# Function to safely combine history logs
def combine_history(h1, h2):
    combined = {}
    # Iterate over all metrics (e.g., 'loss', 'val_loss', 'pr_auc', 'val_pr_auc', etc.)
    for key in h1.history.keys():
        # Concatenate the lists from Stage 1 and Stage 2
        combined[key] = h1.history[key] + h2.history[key]
    return combined

# Create the combined history dictionary
combined_history = combine_history(history_stage1, history_fine)


# --- 2. Plotting Combined History ---

plt.figure(figsize=(14, 5))

# 1. PR-AUC Plot
plt.subplot(1, 2, 1)
# Plot using the combined data
plt.plot(combined_history['pr_auc'], label='Training PR-AUC')
plt.plot(combined_history['val_pr_auc'], label='Validation PR-AUC') 
plt.title('Model PR-AUC Over Epochs (Two Stages)')
plt.xlabel('Epoch')
plt.ylabel('PR-AUC')
plt.axvline(x=len(history_stage1.epoch) - 1, color='gray', linestyle='--', label='End of Stage 1')
plt.legend(loc='lower right')


# 2. Loss Plot
plt.subplot(1, 2, 2)
# Plot using the combined data
plt.plot(combined_history['loss'], label='Training Loss')
plt.plot(combined_history['val_loss'], label='Validation Loss')
plt.title('Model Loss Over Epochs (Two Stages)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.axvline(x=len(history_stage1.epoch) - 1, color='gray', linestyle='--', label='End of Stage 1')
plt.legend(loc='upper right')

plt.tight_layout()
plt.savefig('training_curves_pr_auc_loss_combined.png')
plt.show()

**Figure 5.1** - Training and validation accuracy and loss curves over epochs for the CNN model.

In [ ]:
# Load the best model saved by the callback
print(f"Load the best model from: {CHECKPOINT_PATH}")
best_model = models.load_model(CHECKPOINT_PATH)

# Report metrics
# CORRECTED: Use TEST_DS instead of test_dataset_gray
results = best_model.evaluate(TEST_DS, verbose=2, return_dict=True)

print("\n=== Test Set Evaluation Results ===")
for k,v in results.items():
	print(f"{k}: {v:.4f}")
print("=" * 35)

In [ ]:
# Load the best model saved by the callback
print(f"Load the best model from: {CHECKPOINT_PATH}")
best_model = models.load_model(CHECKPOINT_PATH) # Assuming CHECKPOINT_PATH is defined

print("Generating predictions...")

# Use the correct dataset variable TEST_DS for predictions
y_pred_probs = best_model.predict(TEST_DS, verbose=1).flatten()

# Extract true labels from the dataset
y_true = []
# Use the correct dataset variable TEST_DS to iterate
for _, labels in TEST_DS.unbatch():
    y_true.append(labels.numpy())
y_true = np.array(y_true)

print(f"\nTotal test samples: {len(y_true)}")
print(f"Pneumonia cases: {sum(y_true)} ({sum(y_true)/len(y_true)*100:.1f}%)")
print(f"Normal cases: {len(y_true) - sum(y_true)} ({(len(y_true)-sum(y_true))/len(y_true)*100:.1f}%)")

# Trade-off descriptions helper
def describe_tradeoff(threshold, recall, precision):
    # This function uses recall and precision, but they are not used in the return string.
    # The logic below is maintained for context, but you may want to refine the description.
    if threshold <= 0.2:
        return "Very low threshold; maximize coverage (High Recall, low Precision)"
    elif threshold <= 0.3:
        return "Maximum coverage; few missed cases (High Recall, moderate Precision)"
    elif threshold <= 0.5:
        return "Strong balance (Balanced Recall and Precision)"
    elif threshold <= 0.6:
        return "More conservative (Higher Precision, lower Recall)"
    elif threshold <= 0.8:
        return "Very selective (High Precision, low Recall)"
    else:
        return "Extremely selective; only strongest positives (Highest Precision, lowest Recall)"

# Test thresholds from 0.1 to 0.9 (step 0.1)
thresholds_to_test = [round(t, 1) for t in np.arange(0.1, 1.0, 0.1)]

print("\n" + "="*30)
print("THRESHOLD ANALYSIS")
print("="*30)

results = []
for threshold in thresholds_to_test:
    y_pred = (y_pred_probs >= threshold).astype(int)
    
    # Calculate confusion matrix components
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # Calculate metrics
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    results.append({
        'threshold': threshold,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'recall': recall,
        'precision': precision,
        'accuracy': accuracy,
        'f1': f1
    })
    
    print(f"\nThreshold: {threshold}")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Trade-off: {describe_tradeoff(threshold, recall, precision)}")
    print(f"  ---")
    print(f"  True Positives (Pneumonia correctly identified):  {tp}")
    print(f"  False Negatives (Pneumonia MISSED):               {fn}")
    print(f"  False Positives (Normal incorrectly flagged):     {fp}")
    print(f"  True Negatives (Normal correctly identified):     {tn}")

# Classification Report
y_pred_default = (y_pred_probs >= 0.5).astype(int)
cm = confusion_matrix(y_true, y_pred_default)

print("\nDetailed Classification Report (Threshold = 0.5):")
print(classification_report(
    y_true,
    y_pred_default,
    target_names=['Normal', 'Pneumonia'],
    digits=4
))

&#160;

# 6. Conclusions

---

In [ ]:
# 0.1, 0.2, ..., 0.9
thresholds = [0.1 * i for i in range(1, 10)]

fig, axes = plt.subplots(3, 3, figsize=(12, 10))

for ax, thr in zip(axes.ravel(), thresholds):
    y_pred_thr = (y_pred_probs >= thr).astype(int)
    cm_thr = confusion_matrix(y_true, y_pred_thr)

    sns.heatmap(
        cm_thr,
        annot=True,
        fmt='d',
        cmap='Blues',
        cbar=False,
        xticklabels=['Normal', 'Pneumonia'],
        yticklabels=['Normal', 'Pneumonia'],
        ax=ax
    )

    ax.set_title(f'Threshold = {thr:.1f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.show()

**Figure 6.1** - Confusion matrices on the test set for thresholds

In [ ]:
thresholds = [r['threshold'] for r in results]
recalls = [r['recall'] for r in results]
precisions = [r['precision'] for r in results]
f1_scores = [r['f1'] for r in results]

plt.figure(figsize=(8, 5))

plt.plot(thresholds, recalls, label='Recall')
plt.plot(thresholds, precisions, label='Precision')
plt.plot(thresholds, f1_scores, label='F1-score')

# Highlight recommended threshold
recommended_threshold = 0.6
plt.axvline(recommended_threshold, linestyle='--', label='Recommended threshold (0.6)')

plt.xlabel('Threshold')
plt.ylabel('Metric value')
plt.ylim(0, 1.05)
plt.title("Effect of Threshold on Recall, Precision and F1-score\n")
plt.legend()
plt.tight_layout()
plt.show()

**Figure 6.2** - Recall, precision and F1-score on the test set as a function of threshold.

In [ ]:
# Generate the precision-recall curve data
precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_pred_probs)

# Convert to the right format for plotting (need to reverse the order for proper plotting)
recall_curve = np.flip(recall_curve)
precision_curve = np.flip(precision_curve)

# Precision-Recall curve and Recall/Precision vs Threshold
plt.figure(figsize=(13, 5))

# Left plot: Precision-Recall curve
plt.subplot(1, 2, 1)
plt.plot(recall_curve, precision_curve, linewidth=2, color='#2E86AB')
plt.xlabel('Recall', fontsize=11)
plt.ylabel('Precision', fontsize=11)
plt.title('Precision-Recall Curve\n', fontsize=12)
plt.grid(True, alpha=0.3)

# Mark the tested thresholds (0.1 to 0.9) on the PR curve
for r in results:
    plt.plot(r['recall'], r['precision'], marker='o', markersize=6, color="#E45555")
    plt.annotate(
        f"{r['threshold']:.1f}",
        xy=(r['recall'], r['precision']),
        xytext=(-22, -3),
        textcoords='offset points'
    )

# Right plot: Recall & Precision vs Threshold
plt.subplot(1, 2, 2)
plt.plot(thresholds, recalls, color='#2E86AB', marker='o', linewidth=2, label='Recall', markersize=6)
plt.plot(thresholds, precisions, color='#F18F01', marker='o', linewidth=2, label='Precision', markersize=6)
plt.xlabel('Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Recall & Precision vs Threshold\n', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Figure 6.3** - Precision–recall curve and recall/precision vs threshold on the test set.

___